In [102]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Input
import os

# ======================
# LOAD DATA
# ======================
df = pd.read_csv(r"C:\Users\kuash\Downloads\archive\Gold Price.csv")

df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date').reset_index(drop=True)

data = df[['Price']].values

# ======================
# SPLIT
# ======================
split_raw = int(len(data) * 0.8)

train_data = data[:split_raw]
test_data = data[split_raw:]

# ======================
# SCALE
# ======================
scaler = MinMaxScaler()
train_scaled = scaler.fit_transform(train_data)
test_scaled = scaler.transform(test_data)

# ======================
# SEQUENCES
# ======================
def create_sequences(data, time_step=60):
    X, y = [], []
    for i in range(time_step, len(data)):
        X.append(data[i-time_step:i, 0])
        y.append(data[i, 0])
    return np.array(X), np.array(y)

X_train, y_train = create_sequences(train_scaled, 60)
X_test, y_test = create_sequences(test_scaled, 60)

X_train = X_train.reshape(X_train.shape[0], X_train.shape[1], 1)
X_test = X_test.reshape(X_test.shape[0], X_test.shape[1], 1)

# ======================
# MODEL
# ======================
model = Sequential()
model.add(Input(shape=(60, 1)))
model.add(LSTM(64, return_sequences=True))
model.add(LSTM(32))
model.add(Dense(1))

model.compile(optimizer='adam', loss='mse')

# ======================
# TRAIN
# ======================
model.fit(
    X_train, y_train,
    epochs=30,
    batch_size=32,
    validation_split=0.1,
    verbose=1
)

# ======================
# PREDICT
# ======================
pred = model.predict(X_test)

y_pred_inv = scaler.inverse_transform(pred)
y_test_inv = scaler.inverse_transform(y_test.reshape(-1, 1))

# ======================
# METRICS
# ======================
rmse = np.sqrt(mean_squared_error(y_test_inv, y_pred_inv))
mae = mean_absolute_error(y_test_inv, y_pred_inv)
r2 = r2_score(y_test_inv, y_pred_inv)
mape = np.mean(np.abs((y_test_inv - y_pred_inv) / y_test_inv)) * 100

print("\n=== LSTM MODEL RESULTS (FIXED) ===")
print(f"MAE: {mae:.6f}")
print(f"RMSE: {rmse:.6f}")
print(f"R2: {r2:.6f}")
print(f"MAPE: {mape:.6f}")

# ======================
# SAVE MODEL
# ======================
os.makedirs("models", exist_ok=True)
model.save("models/lstm_model.keras")

Epoch 1/30
69/69 ━━━━━━━━━━━━━━━━━━━━ 8s 49ms/step - loss: 0.0102 - val_loss: 0.0013
Epoch 2/30
69/69 ━━━━━━━━━━━━━━━━━━━━ 3s 42ms/step - loss: 3.7612e-04 - val_loss: 5.3575e-04
Epoch 3/30
69/69 ━━━━━━━━━━━━━━━━━━━━ 3s 37ms/step - loss: 3.6565e-04 - val_loss: 8.5323e-04
Epoch 4/30
69/69 ━━━━━━━━━━━━━━━━━━━━ 3s 42ms/step - loss: 3.5427e-04 - val_loss: 0.0011
Epoch 5/30
69/69 ━━━━━━━━━━━━━━━━━━━━ 3s 44ms/step - loss: 3.6930e-04 - val_loss: 8.4632e-04
Epoch 6/30
69/69 ━━━━━━━━━━━━━━━━━━━━ 3s 40ms/step - loss: 3.4613e-04 - val_loss: 5.8626e-04
Epoch 7/30
69/69 ━━━━━━━━━━━━━━━━━━━━ 3s 40ms/step - loss: 3.2481e-04 - val_loss: 7.4272e-04
Epoch 8/30
69/69 ━━━━━━━━━━━━━━━━━━━━ 3s 40ms/step - loss: 3.2933e-04 - val_loss: 7.8495e-04
Epoch 9/30
69/69 ━━━━━━━━━━━━━━━━━━━━ 3s 41ms/step - loss: 3.6845e-04 - val_loss: 9.0787e-04
Epoch 10/30
69/69 ━━━━━━━━━━━━━━━━━━━━ 3s 42ms/step - loss: 2.9575e-04 - val_loss: 6.3075e-04
Epoch 11/30
69/69 ━━━━━━━━━━━━━━━━━━━━ 3s 40ms/step - loss: 3.0622e-04 - val_loss